# CS-421 Machine learning for behavioral data
## Project - GoGymi dataset
### Fatumah binta Doukouré 340969 - Louis Tschanz 315774 - Majandra Garcia 347470

---

## Sub-question 3
#### To what extent can early learning behaviors, such as exercise diversity, session consistency, and content focus, predict which students will struggle or succeed later in their learning progression?

In [68]:
from utils import *
import math 
from utils import _to_unix, _ms_to_s
import matplotlib
matplotlib.use("Agg")
%load_ext autoreload
%autoreload 2

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import confusion_matrix

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### 1- Load Data 

In [2]:
data_dir = r'C:\Users\fatum\Documents\EPFL\MA4\MLBD\MLBD_2026\gogymi-dataset-2025-2026-complete\out'
tables = load_data(data_dir)

Reading Table students
Reading Table pageviews
Reading Table math_results
Reading Table quiz_results
Reading Table text_results
Reading Table essay_results
Reading Table gymitrainer
Reading Table comments
Reading Table course_ids
Reading Table math_questions
Reading Table text_questions
Reading Table quiz_questions


In [3]:
df_math_results = tables['math_results']
df_math_questions = tables['math_questions']
df_text_results = tables['text_results']
df_quiz_results = tables['quiz_results']
df_quiz_questions = tables['quiz_questions']
df_pageview = tables['pageviews']
df_course_ids = tables['course_ids']
df_students = tables['students']

### 2- Data Mining and Feature Engineering

First, we need to describe the engagement span of the users, to define how long our early window will be 

In [4]:
per_user, early_days = describe_platform_engagement(df_pageview, df_students, early_fraction=1/3)


Unique active days
  Mean   : 28.7
  Median : 23.0
  p25–p75: 9.0 – 42.0

Activity span (days)
  Mean   : 110.1
  Median : 122.9
  p25–p75: 56.0 – 163.8

Days since registration
  Mean   : 130.0
  Median : 134.5
  p25–p75: 89.2 – 176.5

Suggested early window (33% of median span): 41 days


All features are extracted exclusively from each student's first 41 days of engagement

In [5]:
# Filtering dfs for Early days

# Convert timestamps to dt 
df_pageview['created_at'] = pd.to_datetime(df_pageview['created_at'], errors = 'coerce')
df_math_results['timestamp'] = pd.to_datetime(df_math_results['timestamp'], errors = 'coerce')
df_text_results['timestamp'] = pd.to_datetime(df_text_results['timestamp'], errors = 'coerce')
df_quiz_results['time'] = pd.to_datetime(df_quiz_results['time'], errors = 'coerce')

In [6]:
student_meta = df_students[['user_id', 'creation_time']].drop_duplicates()
early_ts = {
    row["user_id"]: row["creation_time"] + early_days * 86400
    for _, row in df_students.iterrows()
}

In [7]:
unique_students_ids = set(df_students['user_id'])
unique_students_ids_in_pageview = set(df_pageview['user_id'])

matching_ids = unique_students_ids  & unique_students_ids_in_pageview

print(f"Matching: {len(matching_ids)}")
print(f"Original number of students: {len(unique_students_ids)}")

Matching: 1691
Original number of students: 1781


In [8]:
print("-"*20 + "Length of dfs before windowing" + "-"*20)
print(f" Pageviews {len(df_pageview)}")
print(f" Math Results {len(df_math_results)}")
print(f" Text Results {len(df_text_results)}")
print(f" Quiz Results {len(df_quiz_results)}")


# Rebuild early_days correctly from UNIX seconds, keeping UTC timezone
df_pageview['early_days'] = pd.to_datetime(df_pageview['user_id'].map(early_ts), unit='s', utc=True)
df_math_results['early_days'] = pd.to_datetime(df_math_results['user_id'].map(early_ts), unit='s')
df_text_results['early_days'] = pd.to_datetime(df_text_results['user_id'].map(early_ts), unit='s')
df_quiz_results['early_days'] = pd.to_datetime(df_quiz_results['user_id'].map(early_ts), unit='s')



# Keep only rows inside the user's first 14 days
early_pageview = df_pageview[df_pageview['created_at'] <= df_pageview['early_days']]
early_math_res = df_math_results[df_math_results['timestamp'] <= df_math_results['early_days']]
early_text_res = df_text_results[df_text_results['timestamp'] <= df_text_results['early_days']]
early_quiz_res = df_quiz_results[df_quiz_results['time'] <= df_quiz_results['early_days']]

print("-"*20 + "Length of dfs After windowing" + "-"*20)

print(f" Pageviews {len(early_pageview)}")
print(f" Math Results {len(early_math_res)}")
print(f" Text Results {len(early_text_res)}")
print(f" Quiz Results {len(early_quiz_res)}")



--------------------Length of dfs before windowing--------------------
 Pageviews 945171
 Math Results 12060
 Text Results 67351
 Quiz Results 239555
--------------------Length of dfs After windowing--------------------
 Pageviews 661760
 Math Results 10159
 Text Results 40700
 Quiz Results 224239


#### Exercise Performance
- Number of unique questions answered by each student

In [9]:
diversity_features = pd.DataFrame({'user_id': df_students['user_id'].unique()})
print(len(diversity_features))

1781


In [10]:
# Unique question ids
math_unique = early_math_res.groupby('user_id')['question_id'].nunique().rename('unique_math_qids')
text_unique = early_text_res.groupby('user_id')['question_id'].nunique().rename('unique_text_qids')
quiz_unique = early_quiz_res.groupby('user_id')['question_id'].nunique().rename('unique_quiz_qids')

diversity_features = diversity_features.merge(math_unique, on='user_id', how='left').fillna(0)
diversity_features = diversity_features.merge(text_unique, on='user_id', how='left').fillna(0)
diversity_features = diversity_features.merge(quiz_unique, on='user_id', how='left').fillna(0)
print(len(diversity_features))

diversity_features

1781


,user_id,unique_math_qids,unique_text_qids,unique_quiz_qids
0,3919,0.0,0.0,0.0
1,359,0.0,0.0,0.0
2,49,0.0,17.0,10.0
3,116,0.0,19.0,172.0
4,118,0.0,18.0,77.0
...,...,...,...,...
1776,6678,0.0,0.0,118.0
1777,6673,0.0,0.0,0.0
1778,6672,0.0,0.0,0.0
1779,6675,0.0,0.0,41.0


#### Shannon Entropy (content focus)
- Quantifies how "spread" the student's attention is accross different content categories

In [11]:
early_pageview['url']  = early_pageview['url'].astype('str') 
df_course_ids['url']  = df_course_ids['url'].astype('str') + "/"

In [12]:
early_pageview = early_pageview.merge(df_course_ids[['url', 'post_type']], on='url', how='left')
print(len(early_pageview))
entropy_features = early_pageview.groupby('user_id')['post_type'].apply(compute_shannon_entropy).rename('content_entropy')

661760


In [13]:
unique_course_url = set(df_course_ids['url'].astype(str))
unique_course_url_in_pageview = set(early_pageview['url'].astype(str))

matching_ids = unique_course_url  & unique_course_url_in_pageview

print(f"Matching: {len(matching_ids)}")
print(f"Original number of urls: {len(unique_course_url)}")

Matching: 403
Original number of urls: 454


#### Time Features

In [15]:
### Temporal Features 
df_ts = build_time_series_features(data_dir, tables, early_ts, n_bins=3)

  Time-series features: 13 features for 1205 users


#### Events Features 
How regularly a student engages with the platform. A student who logs in every day for short focused sessions exhibits a very different learning pattern from one who has one long burst of activity and then disappears, even if their total pageview counts are identical.
Moreover, a student who interacts a lot with the platform (lots of scrolls, pausing videos) might be less focused on the task and impact the learning outcome.
We characterize this through multiple signals: 
- **Total interaction logs** : How much the user interact with the platform
- **Idle time** : The time the user spends in between interactions with the system.
- **active days** The number of distinct calendar days with at least one recorded interaction
- **engagement span**: The time between a student's first and last activity
- **average daily pageviews** normalises volume by active days, distinguishing intensive 


In [16]:
df_features_events = extract_event_features_(data_dir, tables, early_ts)

  Loading event/c.csv  cols=['pageview_id', 'timestamp', 'scrollY']
  Loading event/0.csv  cols=['pageview_id', 'timestamp', 'lastActivity', 'scrollY']
  Loading event/q.csv  cols=['pageview_id', 'timestamp', 'questionNumber']
  Loading event/m.csv  cols=['pageview_id', 'timestamp', 'currentTime', 'duration', 'eventType', 'mediaType', 'mediaId']


In [17]:
outcome = build_outcome(tables, early_ts)
df_ml_events   = df_features_events.merge(outcome[["user_id", "label"]],
                                on="user_id", how="inner")
df_final = df_ml_events
dfs = [diversity_features, entropy_features, df_ts]
for df in dfs: 
    print(f"---- Length before merge {len(df_final)}")
    df_final = df_final.merge(df, on='user_id', how='left')
    print(f"---- Length after merge {len(df_final)}")
df_final  = df_final.dropna(subset=["label"])


 Outcome median score: 0.587
---- Length before merge 575
---- Length after merge 575
---- Length before merge 575
---- Length after merge 575
---- Length before merge 575
---- Length after merge 575


In [18]:
# Convert timedelta to float64 (seconds)
if "heartbeat__avg_idle_time" in df_final.columns:
    if pd.api.types.is_timedelta64_dtype(df_final["heartbeat__avg_idle_time"]):
        df_final["heartbeat__avg_idle_time"] = (
            df_final["heartbeat__avg_idle_time"].dt.total_seconds().astype("float64")
        )
    else:
        df_final["heartbeat__avg_idle_time"] = pd.to_numeric(
            df_final["heartbeat__avg_idle_time"], errors="coerce"
        ).astype("float64")

In [ ]:
feature_cols = [c for c in df_final.columns if c not in ("user_id", "label")]

### Analysis per group of features 
Now that we have all of these features, we can separate them into multiple groups and see how they add predictive power to our model. They will be separated into 5 groups and evaluate the model on each one of them before evaluating it on all of them.
- Engagement Volume:clicks__n_total, heartbeat__n_total, q__n_questions_viewed, media__n_play_events, media__total_watch_min
- Content Breadth: q__n_unique_urls, q__n_unique_courses, media__n_unique_urls, media__n_unique_media, media__media_type_diversity, unique_math_qids, unique_text_qids, unique_quiz_qids
- Session Consistency: clicks__session_gap_cv, heartbeat__avg_idle_time, q__n_active_days
- Physical Attention (Scroll Depth) : clicks__avg_scrollY, clicks__scroll_depth_max, heartbeat__avg_scrollY, heartbeat__scroll_depth_max
- Learning Stategy: content_entropy, q__avg_question_number, q__revisit_rate, q__maj_track, media__audio_pct,  media__completion_rate
- Time Series 


#### Correlation Analysis 

Let's start by vizualizing for each group the correlation matrices, and prune the redundant features.

In [22]:
EVENT_FEATURE_GROUPS = {
    "Engagement Volume": [
        "clicks__n_total", "heartbeat__n_total", "q__n_questions_viewed",
        "media__n_play_events", "media__total_watch_min",
    ],
    "Content Breadth": [
        "q__n_unique_urls", "q__n_unique_courses", "media__n_unique_urls",
        "media__n_unique_media", "media__media_type_diversity",
        "unique_math_qids", "unique_text_qids", "unique_quiz_qids",
    ],
    "Session Regularity": [
        "clicks__session_gap_cv", "heartbeat__avg_idle_time", "q__n_active_days",
    ],
    #"Scroll Depth": [
    #    "clicks__avg_scrollY", "clicks__scroll_depth_max",
    #    "heartbeat__avg_scrollY", "heartbeat__scroll_depth_max",
    #],
    "Learning Strategy": [
        "content_entropy", "q__avg_question_number", "q__revisit_rate",
        "q__maj_track", "media__audio_pct", "media__completion_rate",
    ],
    "Timeseries" :  [
    "ts__q_bin1", "ts__q_bin2", "ts__q_bin3",
    "ts__q_trend", "ts__q_recency",
    "ts__pv_bin1", "ts__pv_bin2", "ts__pv_bin3",
    "ts__pv_trend", "ts__pv_recency",
    "ts__score_first", "ts__score_last", "ts__score_delta",
]
}

In [23]:
for group_name, features in EVENT_FEATURE_GROUPS.items():
    plot_correlation_heatmap(df_final, features, group_name)

Engagement Volume
Content Breadth
Session Regularity
Learning Strategy
Timeseries


![Correlation CB](Correlation_Matrices\Correlation%20matrix%20for%20Content%20Breadth.png)
![Correlation_EV](Correlation_Matrices\Correlation%20matrix%20for%20Engagement%20Volume.png)
![Correlation CB](Correlation_Matrices\Correlation%20matrix%20for%20Learning%20Strategy.png)
![Correlation CB](Correlation_Matrices\Correlation%20matrix%20for%20Session%20Regularity.png)
![Correlation CB](Correlation_Matrices\Correlation%20matrix%20for%20Time%20Series.png)

Inside all of the groups, the correlations between features are moderate (none >0.8). Pruning will thus not be necessary before the ablation study. 

#### Ablation Study 

In [24]:
evaluate_by_group(df_final, EVENT_FEATURE_GROUPS)

  [Engagement Volume]... AUC=0.527
  [Content Breadth]... AUC=0.678
  [Session Regularity]... AUC=0.548
  [Learning Strategy]... AUC=0.547
  [Timeseries]... AUC=0.597
  [All Combined]... AUC=0.684

── Group Ablation Study ─────────────────────────────────────
  Group                     N       ROC-AUC      F1   Bal Acc
  ────────────────────────────────────────────────────────────
  Engagement Volume         5  0.527±0.047  0.485  0.509
  Content Breadth           8  0.678±0.043  0.596  0.615
  Session Regularity        3  0.548±0.044  0.550  0.548
  Learning Strategy         6  0.547±0.014  0.533  0.520
  Timeseries               13  0.597±0.048  0.543  0.561
  All Combined             35  0.684±0.050  0.626  0.628


{'Engagement Volume': {'roc_auc': 0.5271347934418161,
  'roc_auc_std': 0.04744021340091061,
  'f1': 0.4854920979932624,
  'bal_acc': 0.5092230320101833,
  'n_features': 5},
 'Content Breadth': {'roc_auc': 0.6783342781266433,
  'roc_auc_std': 0.042531082661341424,
  'f1': 0.5958494962908025,
  'bal_acc': 0.6152904543650183,
  'n_features': 8},
 'Session Regularity': {'roc_auc': 0.5480729512266932,
  'roc_auc_std': 0.04360087441056782,
  'f1': 0.5502112734220833,
  'bal_acc': 0.5480115945913878,
  'n_features': 3},
 'Learning Strategy': {'roc_auc': 0.547372247831736,
  'roc_auc_std': 0.01391252513382698,
  'f1': 0.5328961948932143,
  'bal_acc': 0.5204138341177315,
  'n_features': 6},
 'Timeseries': {'roc_auc': 0.5969338161300387,
  'roc_auc_std': 0.04830085064842289,
  'f1': 0.5431134782437798,
  'bal_acc': 0.5614782316691788,
  'n_features': 13},
 'All Combined': {'roc_auc': 0.684295301374418,
  'roc_auc_std': 0.04999554624924412,
  'f1': 0.6264728195491484,
  'bal_acc': 0.6280219104160

![Ablation](\group_ablation.png)

The ablation study shows content breadth is the most predictive  feature group, implying that early content diversity is a strong behavioral signal.

#### Multi-model comparison

In [50]:
X = df_final[feature_cols].copy()
nan_cols = X.columns[X.isna().all()].tolist()
X = X.drop(columns=nan_cols)
y = df_final["label"].astype(int)
print(f"\n  Features used : {len(feature_cols)}")
print(f"  Sample size   : {len(X)}")
print(f"  Class balance : {y.value_counts().to_dict()}")


  Features used : 40
  Sample size   : 575
  Class balance : {0: 291, 1: 284}


In [51]:
results = evaluate_models(X, y, None)
plot_feature_importance(results, X, 'Feature_importance.png')


── Cross-validated performance ───────────────────────────
  Model                       ROC-AUC        F1  Balanced Acc
  ------------------------------------------------------------
  Logistic Regression       0.668±0.043  0.605  0.607
  Random Forest             0.687±0.045  0.608  0.622
  Gradient Boosting         0.694±0.033  0.632  0.630
  XGBoost                   0.687±0.041  0.623  0.634


We were able to obtain better than random performance on all of our models. No single model is dominating convincingly, thus we cannot infer on the linear nature of our features' interaction. The results are then mostly influenced by the feature engineering.

![Feature Importance](Feature_importance.png)

Through this feature importance plot, we make interesting findings:

- unique_quiz_qids and unique_math_qids rank at the top across all four models, making it a robust predictor.
- In Linear regression, unique_quiz_qids strongly predicts success : students who engage with a diverse range of quiz content early on tend to perform better
- Surprisingly, unique_math_qids strongly predicts struggle Unlike quiz diversity, math question diversity is a negative signal, suggesting students who scatter across many math topics without consolidation may be struggling to find their footing rather than exploring productively
- q_n_active_days predicts success: regular platform presence over the early window is a positive signal
- q_revisit_rate, clicks_n_total, q_n_unique_urls, q_n_questions_viewed all predict struggle, showing that high volume and repetitive revisiting are signs of difficulty, not engagement quality
- clicks_session_gap_cv predicts success, which could mean that student that interact more sparsely with the platform are more focused on solving the problems.
- Media features are consistently irrelevant
- GB and XGBoost surface content_entropy,  suggesting that it contributes through non-linear interactions 

What about with a black box model ? 

In [ ]:
lstm_cols = [
    "ts__q_bin1", "ts__pv_bin1",   
    "ts__q_bin2", "ts__pv_bin2", 
    "ts__q_bin3", "ts__pv_bin3",
]

X_lstm = df_final[lstm_cols]
y = df_final["label"].astype(int)

loaded_model = train_bidirectional_lstm(X_lstm, y, num_weeks=3)


Epoch 1/10
12/12 [==============================] - 28s 824ms/step - loss: 0.6899 - accuracy: 0.5217 - val_loss: 0.6837 - val_accuracy: 0.6304
Epoch 2/10
12/12 [==============================] - 1s 127ms/step - loss: 0.6911 - accuracy: 0.5408 - val_loss: 0.6825 - val_accuracy: 0.6413
Epoch 3/10
12/12 [==============================] - 0s 38ms/step - loss: 0.6897 - accuracy: 0.5652 - val_loss: 0.6801 - val_accuracy: 0.6413
Epoch 4/10
12/12 [==============================] - 0s 33ms/step - loss: 0.6914 - accuracy: 0.5543 - val_loss: 0.6781 - val_accuracy: 0.6304
Epoch 5/10
12/12 [==============================] - 0s 31ms/step - loss: 0.6906 - accuracy: 0.5326 - val_loss: 0.6798 - val_accuracy: 0.6522
Epoch 6/10
12/12 [==============================] - 1s 59ms/step - loss: 0.6882 - accuracy: 0.5652 - val_loss: 0.6801 - val_accuracy: 0.6522
Epoch 7/10
12/12 [==============================] - 0s 35ms/step - loss: 0.6864 - accuracy: 0.5435 - val_loss: 0.6810 - val_accuracy: 0.6413
Epoch 8/10

### Vizualisation of the feature space using KMeans and prediction with clusters as a feature

Prior work has shown that learners naturally segment into a small number of behaviorally distinct profiles.(Kizilcec, R. F., Piech, https://doi.org/10.1145/2460296.2460330). They established that unsupervised clustering makes subpopulation appear.

We apply K-Means (k=3) to our feature space and project the results onto the first two principal components to assess separability. 

In [52]:
#features_list = ['unique_math_qids', 'unique_text_qids', 'content_entropy', 'unique_quiz_qids']
X = df_final[feature_cols].fillna(0) # Handle missing values
X = X.astype('float64')
# 2. Standardize the data
scaler = StandardScaler()
groups = {
    "clicks":    ["clicks__n_total", "clicks__session_gap_cv", "clicks__scroll_depth_max"],
    "exercise":  ["unique_quiz_qids", "unique_math_qids", "unique_text_qids", "q__revisit_rate"],
    "content":   ["content_entropy", "q__n_unique_courses"],
    "media":     ["media__n_play_events", "media__completion_rate", "media__audio_pct"],
}

scaled_parts = []
for name, cols in groups.items():
    available = [c for c in cols if c in df_final.columns]
    scaled_parts.append(pd.DataFrame(
        scaler.fit_transform(df_final[available].fillna(0)),
        columns=available
    ))


X_scaled = pd.concat(scaled_parts, axis=1)


kmeans = KMeans(n_clusters=3, random_state=42)
clusters = kmeans.fit_predict(X_scaled)
df_final['cluster'] = clusters


pca = PCA(n_components=2)
pca_results = pca.fit_transform(X_scaled)
df_final['pca1'] = pca_results[:, 0]
df_final['pca2'] = pca_results[:, 1]


plt.figure(figsize=(10, 7))
sns.scatterplot(
    x='pca1', y='pca2', 
    hue='cluster', 
    palette='viridis', 
    data=df_final, 
    s=100, alpha=0.7
)

plt.title('Student Behavioral Profiles: Clustering by Early Learning Metrics')
plt.xlabel('Principal Component 1 ')
plt.ylabel('Principal Component 2')
plt.legend(title='Student Group')
plt.savefig("PCA.jpg")

![Feature Importance]( \PCA.jpg)

We observe three profiles: 
- A large group (Yellow) clustered around the origin
- A blue group spread along the positive PC1 axis.
- A small group (Purple) isolated in the positive region.

Let's dig more into what these clusters mean about student behavior.


In [66]:

key_features = [
    "unique_quiz_qids", "unique_math_qids", "q__n_active_days",
    "q__n_questions_viewed", "content_entropy",
    "clicks__session_gap_cv", "clicks__n_total", "media__n_unique_urls", "q__revisit_rate", "media__completion_rate"
]
cluster_means = df_final.groupby("cluster")[key_features].mean()

cluster_means_norm = (cluster_means - cluster_means.mean()) / cluster_means.std()

sns.heatmap(cluster_means_norm.T, annot=False, fmt=".2f", cmap="coolwarm",
         linewidths=0.5)
plt.title("Cluster Profiles")
plt.xlabel("Cluster")
plt.savefig("cluster_profile.png")


![Cluster Profile](\cluster_profile.png)

This profiling gives interesting insights of the behavior of the students in each clusters.
- Group 0 is bellow average on everything, reflecting disengaged students, that show low activity, low diversity and low regularity in the beginning.
- Group 1 show highly engaged students accross all dimensions.
- Group 2 is a cluster of moderate users.

In [63]:
print("------------------ Outcome Rate ------------------")
print(df_final.groupby("cluster").agg(
    n_students   = ("label", "count"),
    success_rate = ("label", "mean"),
    pct_langzeit = ("q__maj_track", "mean")
).round(3))

------------------ Outcome Rate ------------------
         n_students  success_rate  pct_langzeit
cluster                                        
0                33         0.667         1.000
1               219         0.507         0.420
2               323         0.467         0.353


Surprisingly, the group of disengaged students seem to have the highest success rate, while the moderate ones have the lowest one.
One interesting finding here is that the cluster success rate ordering mirrors the Langzeit representation ordering exactly, suggesting that behavioral engagement on this platform is reflects partially the track of the students. This raises a fundamental question: does our model predict learning success from behavior, or does it predict track membership, which correlates with success through socioeconomic mechanisms outside the platform?

### Ethics: Are the model equally performant for langzeit and kurzeit reflecting socio-economical status
We will start by investigate if the  outcomes equally predictable within each track ? 

We will thenuse the model with the best performances, XGBoost and try to answer the following questions:
1. Does the model make equal errors per track : Per group evaluation
2. Which group gets systematically harmed by the model : FNR, FPR

In [84]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
pipe = build_models()["Gradient Boosting"]

kurz_mask = (df_final['q__maj_track'] == 0).values
lang_mask = (df_final['q__maj_track'] == 1).values

kurz_aucs, lang_aucs = [], []

oof_proba = np.zeros(len(X)) 

for train_idx, test_idx in cv.split(X, y):
    pipe.fit(X.iloc[train_idx], y.iloc[train_idx])
    y_prob = pipe.predict_proba(X.iloc[test_idx])[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)
    y_true = y.iloc[test_idx].values

    oof_proba[test_idx] = y_prob
    # Which test indices belong to each track
    kurz_fold = kurz_mask[test_idx]
    lang_fold = lang_mask[test_idx]

    if kurz_fold.any():
        kurz_aucs.append(roc_auc_score(y_true[kurz_fold], y_prob[kurz_fold]))
        
    if lang_fold.any():
        lang_aucs.append(roc_auc_score(y_true[lang_fold], y_prob[lang_fold]))
        

df_final["pred_proba"] = oof_proba
df_final["pred_label"] = (oof_proba >= 0.5).astype(int)

print(f"Kurzeit — AUC: {np.mean(kurz_aucs):.3f}")
print(f"Langzeit — AUC: {np.mean(lang_aucs):.3f}")


Kurzeit — AUC: 0.715
Langzeit — AUC: 0.587


- We observe a substantial AUC gap. Early behaviors are less informative predictors of later success for langzeit students 


In [81]:
def fairness_metrics(df_kurz, df_lang):
    
    # Confusion Matrix
    cm_kurz = confusion_matrix(df_kurz['label'],df_kurz['pred_label'])
    cm_lang = confusion_matrix(df_lang['label'],df_lang['pred_label'])
    TN_k, FP_k, FN_k, TP_k = cm_kurz.ravel()
    TN_l, FP_l, FN_l, TP_l = cm_lang.ravel()

    
    # Total population
    N_k = TP_k + FP_k + FN_k + TN_k
    N_l = TP_l + FP_l + FN_l + TN_l 

    
    # True positive rate
    tpr_k = TP_k / (TP_k + FN_k) 
    tpr_l = TP_l / (TP_l + FN_l) 


    # Percentage predicted as positive
    ppp_k = (TP_k+FP_k)/N_k
    ppp_l = (TP_l+FP_l)/N_l


    # FNR
    fnr_k = FN_k/(FN_k+TP_k)
    fnr_l = FN_l/(FN_l+TP_l)


    # FPR
    fpr_k = FP_k/(FP_k+TN_k)
    fpr_l = FP_l/(FP_l+TN_l)


    metrics = {
    "TPR : Equal Opportunity":  (tpr_k, tpr_l),
    "FPR : False Positive Rate":   (fpr_k, fpr_l),
    "PPP : Disparate Impact": (ppp_k, ppp_l),
    "FNR : False Negative Rate" : (fnr_k, fnr_l)
    }

    return metrics

In [82]:
metrics = fairness_metrics(df_final[df_final['q__maj_track'] == 0], df_final[df_final['q__maj_track'] == 1])

In [86]:
rows = []
for name, (k, l) in metrics.items():
    rows.append({"Metric": name, "Kurzeit": k, "Langzeit": l, "Gap": abs(k - l)})

results_df = pd.DataFrame(rows).set_index("Metric")
results_df.style\
    .format({"Kurzeit": "{:.3f}", "Langzeit": "{:.3f}", "Gap": "{:.3f}"})\
    .set_caption("Fairness Metrics by Track")

,Kurzeit,Langzeit,Gap
Metric,,,
TPR : Equal Opportunity,0.686,0.495,0.191
FPR : False Positive Rate,0.390,0.371,0.020
PPP : Disparate Impact,0.542,0.427,0.115
FNR : False Negative Rate,0.314,0.505,0.191


In [87]:
print(f"PPP ratio = {0.427/0.542:.2f}")

PPP ratio = 0.79


- The model shows significant discrepancies between the two tracks. The FNR gap of 0.191 shows that more than half of Langzeit students are incorrectly classified as strugglin against only 31% of kurzeit. The low FPR gap tells us that the model does not harm struggling students but fails to recognize successful ones in the lagzeit track.